In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

configs = {
    "1 pod":  "results/1pod_stats_history.csv",
    "2 pods": "results/2pod_stats_history.csv",
    "4 pods": "results/4pod_stats_history.csv",
    "8 pods": "results/8pod_stats_history.csv",
}
colors = ["steelblue", "darkorange", "green", "tomato"]
breaking_pts = {"1 pod": 35, "2 pods": 80, "4 pods": 130, "8 pods": 166}

def load_history(path, value_col, max_val=None):
    df = pd.read_csv(path)
    df = df[(df["Name"] == "Aggregated") & (df["User Count"] > 0)]
    df = df.replace("N/A", float("nan")).dropna(subset=[value_col])
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df["Failures/s"] = pd.to_numeric(df["Failures/s"], errors="coerce").fillna(0)
    # keep only stable (no-failure) intervals to avoid timeout spikes
    df = df[df["Failures/s"] == 0]
    if max_val is not None:
        df = df[df[value_col] < max_val]
    grouped = df.groupby("User Count")[value_col].mean()
    return grouped

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Average Response Time vs Concurrent Users
for (label, path), color in zip(configs.items(), colors):
    s = load_history(path, "Total Average Response Time") / 1000  # ms -> s
    axes[0].plot(s.index, s.values, "-", color=color, linewidth=2, label=label)
    bp = breaking_pts[label]
    axes[0].axvline(x=bp, color=color, linestyle="--", linewidth=0.8, alpha=0.5)
axes[0].set_xlabel("Concurrent Users")
axes[0].set_ylabel("Avg Response Time (s)")
axes[0].set_title("Average Response Time vs Concurrent Users")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 35)

# Plot 2: Throughput (RPS) vs Concurrent Users
for (label, path), color in zip(configs.items(), colors):
    s = load_history(path, "Requests/s")
    axes[1].plot(s.index, s.values, "-", color=color, linewidth=2, label=label)
    bp = breaking_pts[label]
    axes[1].axvline(x=bp, color=color, linestyle="--", linewidth=0.8, alpha=0.5)
axes[1].set_xlabel("Concurrent Users")
axes[1].set_ylabel("Throughput (req/s)")
axes[1].set_title("Throughput vs Concurrent Users")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../benchmark_plots.png", dpi=150, bbox_inches="tight")
plt.show()


In [2]:
!pip list | grep numpy

numpy                       2.4.4
